## Script de sauvegarde et chargement du modèle

In [ ]:
from keras.models import save_model, load_model

def save_and_load_model(model, model_path='imdb_sentiment_model.keras'):
    """
    Sauvegarde et charge le modèle
    """
    # Sauvegarde
    save_model(model, model_path)
    print(f"Modèle sauvegardé dans {model_path}")
    
    # Chargement
    loaded_model = load_model(model_path)
    print("Modèle chargé avec succès!")
    
    return loaded_model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from keras.datasets import imdb
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.optimizers import RMSprop
from keras.regularizers import l2
import warnings
warnings.filterwarnings('ignore')

# Configuration pour la reproductibilité
np.random.seed(42)

# ==================== ÉTAPE 1: CHARGEMENT DES DONNÉES ====================

def load_and_prepare_data(num_words=10000, maxlen=500):
    """
    Charge et prépare l'ensemble de données IMDB
    """
    print("="*60)
    print("CHARGEMENT DE L'ENSEMBLE DE DONNÉES IMDB")
    print("="*60)
    
    # Chargement des données
    (train_data, train_labels), (test_data, test_labels) = imdb.load_data(
        num_words=num_words
    )
    
    print(f"Taille des données d'entraînement: {len(train_data)}")
    print(f"Taille des données de test: {len(test_data)}")
    print(f"Nombre de mots uniques: {num_words}")
    
    # Limiter la longueur des séquences pour le traitement
    train_data = [seq[:maxlen] for seq in train_data]
    test_data = [seq[:maxlen] for seq in test_data]
    
    # Afficher des exemples
    print("\nExemple de critique encodée (les 20 premiers mots):")
    print(train_data[0][:20])
    print(f"Étiquette: {'Positive' if train_labels[0] == 1 else 'Negative'}")
    
    return train_data, train_labels, test_data, test_labels

# ==================== ÉTAPE 2: VECTORISATION DES DONNÉES ====================

def vectorize_sequences(sequences, dimension=10000):
    """
    Convertit les séquences d'entiers en matrices binaires one-hot
    """
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1
    return results

def prepare_data(train_data, train_labels, test_data, test_labels, num_words=10000):
    """
    Vectorise les données et les divise en ensembles
    """
    print("\n" + "="*60)
    print("VECTORISATION DES DONNÉES")
    print("="*60)
    
    # Vectorisation
    x_train = vectorize_sequences(train_data, num_words)
    x_test = vectorize_sequences(test_data, num_words)
    y_train = np.asarray(train_labels).astype('float32')
    y_test = np.asarray(test_labels).astype('float32')
    
    print(f"Forme des données d'entraînement vectorisées: {x_train.shape}")
    print(f"Forme des données de test vectorisées: {x_test.shape}")
    
    # Division en ensemble de validation (prendre 25% des données d'entraînement)
    validation_size = int(0.25 * len(x_train))
    
    x_val = x_train[:validation_size]
    y_val = y_train[:validation_size]
    x_train_partial = x_train[validation_size:]
    y_train_partial = y_train[validation_size:]
    
    print(f"Taille de l'ensemble de validation: {len(x_val)}")
    print(f"Taille de l'ensemble d'entraînement: {len(x_train_partial)}")
    
    return x_train_partial, x_val, x_test, y_train_partial, y_val, y_test

# ==================== ÉTAPE 3: CONSTRUCTION DU MODÈLE ====================

def build_model(input_dim=10000, dropout_rate=0.5, l2_reg=0.001):
    """
    Construit le modèle de réseau neuronal
    """
    print("\n" + "="*60)
    print("CONSTRUCTION DU MODÈLE")
    print("="*60)
    
    model = Sequential([
        # Première couche cachée
        Dense(64, activation='relu', input_shape=(input_dim,), 
              kernel_regularizer=l2(l2_reg)),
        Dropout(dropout_rate),
        
        # Deuxième couche cachée
        Dense(32, activation='relu', kernel_regularizer=l2(l2_reg)),
        Dropout(dropout_rate),
        
        # Couche de sortie
        Dense(1, activation='sigmoid')
    ])
    
    # Compilation du modèle
    model.compile(
        optimizer=RMSprop(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    print("Architecture du modèle:")
    model.summary()
    
    return model

# ==================== ÉTAPE 4: ENTRAÎNEMENT DU MODÈLE ====================

def train_model(model, x_train, y_train, x_val, y_val, epochs=20, batch_size=512):
    """
    Entraîne le modèle et retourne l'historique
    """
    print("\n" + "="*60)
    print("ENTRAÎNEMENT DU MODÈLE")
    print("="*60)
    
    history = model.fit(
        x_train, y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(x_val, y_val),
        verbose=1
    )
    
    return history

# ==================== ÉTAPE 5: VISUALISATION DES RÉSULTATS ====================

def plot_training_history(history, title_prefix=""):
    """
    Visualise les courbes de perte et de précision
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Perte
    epochs = range(1, len(history.history['loss']) + 1)
    ax1.plot(epochs, history.history['loss'], 'b-', label='Perte entraînement', linewidth=2)
    ax1.plot(epochs, history.history['val_loss'], 'r-', label='Perte validation', linewidth=2)
    ax1.set_xlabel('Époques', fontsize=12)
    ax1.set_ylabel('Perte', fontsize=12)
    ax1.set_title(f'{title_prefix}Perte - Entraînement vs Validation', fontsize=14)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Précision
    ax2.plot(epochs, history.history['accuracy'], 'b-', label='Précision entraînement', linewidth=2)
    ax2.plot(epochs, history.history['val_accuracy'], 'r-', label='Précision validation', linewidth=2)
    ax2.set_xlabel('Époques', fontsize=12)
    ax2.set_ylabel('Précision', fontsize=12)
    ax2.set_title(f'{title_prefix}Précision - Entraînement vs Validation', fontsize=14)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('imdb_training_history.png', dpi=300, bbox_inches='tight')
    plt.show()

# ==================== ÉTAPE 6: DÉTECTION DU SURAPPRENTISSAGE ====================

def detect_overfitting(history):
    """
    Détecte le surapprentissage et retourne le nombre optimal d'époques
    """
    val_loss = history.history['val_loss']
    val_accuracy = history.history['val_accuracy']
    
    # Trouver l'époque avec la meilleure perte de validation
    best_epoch = np.argmin(val_loss) + 1
    best_val_loss = min(val_loss)
    best_val_accuracy = max(val_accuracy)
    
    print("\n" + "="*60)
    print("ANALYSE DU SURAPPRENTISSAGE")
    print("="*60)
    print(f"Meilleure perte de validation: {best_val_loss:.4f} à l'époque {best_epoch}")
    print(f"Meilleure précision de validation: {best_val_accuracy:.4f} à l'époque {best_epoch}")
    
    # Détection du surapprentissage
    train_loss = history.history['loss']
    train_accuracy = history.history['accuracy']
    
    # Calculer les différences
    loss_diff = [train_loss[i] - val_loss[i] for i in range(len(train_loss))]
    acc_diff = [train_accuracy[i] - val_accuracy[i] for i in range(len(train_accuracy))]
    
    # Identifier les époques où le surapprentissage commence
    overfit_start = None
    for i in range(1, len(loss_diff)):
        if loss_diff[i] < loss_diff[i-1] and val_loss[i] > val_loss[i-1]:
            overfit_start = i
            break
    
    if overfit_start:
        print(f"Le surapprentissage commence à l'époque {overfit_start + 1}")
    else:
        print("Pas de surapprentissage évident détecté")
    
    return best_epoch

# ==================== ÉTAPE 7: ÉVALUATION FINALE ====================

def evaluate_final_model(model, x_test, y_test):
    """
    Évalue le modèle final sur l'ensemble de test
    """
    print("\n" + "="*60)
    print("ÉVALUATION FINALE SUR L'ENSEMBLE DE TEST")
    print("="*60)
    
    test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
    
    print(f"Perte sur l'ensemble de test: {test_loss:.4f}")
    print(f"Précision sur l'ensemble de test: {test_accuracy:.4f}")
    print(f"Précision en pourcentage: {test_accuracy * 100:.2f}%")
    
    return test_loss, test_accuracy

# ==================== ÉTAPE 8: PRÉDICTIONS ET ANALYSE ====================

def analyze_predictions(model, x_test, y_test, num_samples=10):
    """
    Analyse les prédictions du modèle sur quelques exemples
    """
    print("\n" + "="*60)
    print("ANALYSE DES PRÉDICTIONS")
    print("="*60)
    
    # Faire des prédictions
    predictions = model.predict(x_test[:num_samples], verbose=0)
    
    print("\nExemples de prédictions:")
    print("-" * 60)
    for i in range(num_samples):
        pred_class = 1 if predictions[i] >= 0.5 else 0
        true_class = y_test[i]
        confidence = predictions[i][0] if pred_class == 1 else 1 - predictions[i][0]
        
        status = "✅ Correct" if pred_class == true_class else "❌ Incorrect"
        sentiment = "Positive" if pred_class == 1 else "Négative"
        
        print(f"Échantillon {i+1}: Prédiction={sentiment} ({confidence:.2%}), "
              f"Réel={'Positive' if true_class == 1 else 'Négative'}, {status}")

# ==================== ÉTAPE 9: MODÈLE OPTIMAL ====================

def train_optimal_model(x_train, y_train, x_val, y_val, x_test, y_test, best_epoch):
    """
    Réentraîne le modèle avec le nombre optimal d'époques
    """
    print("\n" + "="*60)
    print(f"RÉENTRAÎNEMENT DU MODÈLE OPTIMAL ({best_epoch} ÉPOQUES)")
    print("="*60)
    
    # Créer un nouveau modèle
    optimal_model = build_model()
    
    # Entraîner avec le nombre optimal d'époques
    history_optimal = optimal_model.fit(
        x_train, y_train,
        epochs=best_epoch,
        batch_size=512,
        validation_data=(x_val, y_val),
        verbose=1
    )
    
    return optimal_model, history_optimal

# ==================== FONCTION PRINCIPALE ====================

def main():
    """
    Fonction principale exécutant tout le pipeline
    """
    # 1. Chargement des données
    train_data, train_labels, test_data, test_labels = load_and_prepare_data()
    
    # 2. Préparation des données
    x_train, x_val, x_test, y_train, y_val, y_test = prepare_data(
        train_data, train_labels, test_data, test_labels
    )
    
    # 3. Construction du modèle initial
    model = build_model()
    
    # 4. Entraînement initial
    history = train_model(model, x_train, y_train, x_val, y_val, epochs=20)
    
    # 5. Visualisation
    plot_training_history(history, "Modèle Initial - ")
    
    # 6. Détection du surapprentissage
    best_epoch = detect_overfitting(history)
    
    # 7. Réentraînement avec le nombre optimal d'époques
    optimal_model, history_optimal = train_optimal_model(
        x_train, y_train, x_val, y_val, x_test, y_test, best_epoch
    )
    
    # 8. Visualisation du modèle optimal
    plot_training_history(history_optimal, "Modèle Optimal - ")
    
    # 9. Évaluation finale
    test_loss, test_accuracy = evaluate_final_model(optimal_model, x_test, y_test)
    
    # 10. Analyse des prédictions
    analyze_predictions(optimal_model, x_test, y_test)
    
    # 11. Résumé final
    print("\n" + "="*60)
    print("RÉSUMÉ FINAL")
    print("="*60)
    print(f"✅ Modèle entraîné avec succès!")
    print(f"✅ Précision sur l'ensemble de test: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
    print(f"✅ Perte sur l'ensemble de test: {test_loss:.4f}")
    print(f"✅ Nombre optimal d'époques: {best_epoch}")
    print("✅ Visualisations sauvegardées dans 'imdb_training_history.png'")
    print("="*60)

if __name__ == "__main__":
    main()

In [ ]:
def predict_sentiment(text, model, word_index, num_words=10000, maxlen=500):
    """
    Prédit le sentiment d'un nouveau texte
    """
    # Prétraitement du texte
    words = text.lower().split()
    sequence = []
    
    for word in words:
        if word in word_index and word_index[word] < num_words:
            sequence.append(word_index[word])
    
    # Limiter la longueur
    sequence = sequence[:maxlen]
    
    # Vectorisation
    vectorized = vectorize_sequences([sequence], num_words)
    
    # Prédiction
    prediction = model.predict(vectorized)[0][0]
    sentiment = "POSITIVE" if prediction >= 0.5 else "NÉGATIVE"
    confidence = prediction if prediction >= 0.5 else 1 - prediction
    
    print(f"\nTexte: {text}")
    print(f"Sentiment prédit: {sentiment} ({confidence:.2%} de confiance)")
    
    return prediction, sentiment

# Exemple d'utilisation (à placer après l'importation du word_index)
# word_index = keras.datasets.imdb.get_word_index()
# predict_sentiment("This movie was fantastic! Great acting and storyline.", model, word_index)